In [19]:
#单环境
import json
import networkx as nx
import numpy as np
import random
import os
from collections import Counter, defaultdict
from pathlib import Path

class ToolChainGenerator:
    def __init__(self, graph_path, steps = None):
        self.graph_path = graph_path
        self.file_name = os.path.basename(graph_path)
        
        with open(graph_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        self.G = nx.node_link_graph(data, directed=True)
        self.nodes = list(self.G.nodes())
        self.num_nodes = self.G.number_of_nodes()
        self.steps = self.num_nodes + 3 if steps is None else steps

    def _softmax(self, weights, temperature=1.0):
        weights = np.array(weights)
        e_x = np.exp((weights - np.max(weights)) / temperature)
        return e_x / e_x.sum()

    def generate(self, num_walks_per_node=3, temperature=1.0, 
                 max_repeats=3, base_stop_prob=0.05, stop_steps = 5):
        """
        :param max_repeats: 单个节点允许的最大重复次数 (硬刹车)
        :param base_stop_prob: 每一步的基础停止概率 (软刹车)
        """
        chains = []
        isolates = list(nx.isolates(self.G))
        for node in isolates:
            chains.append([node])

        active_nodes = [n for n in self.nodes if n not in isolates]
        
        # 绝对上限仍然是节点总数 (防止极端情况)
        abs_max_len = self.steps

        for start_node in active_nodes:
            for _ in range(num_walks_per_node):
                chain = [start_node]
                curr = start_node
                
                # 使用 Counter 实时跟踪当前链的节点重复情况
                node_counts = defaultdict(int)
                node_counts[curr] += 1
                
                while len(chain) < abs_max_len:
                    # 1. 获取所有邻居
                    neighbors = list(self.G.successors(curr))
                    if not neighbors:
                        break # 死胡同，自然停止

                    # 2. 【硬刹车】过滤掉重复次数超标的节点
                    valid_neighbors = [n for n in neighbors if node_counts[n] < max_repeats]
                    
                    if not valid_neighbors:
                        break # 周围的节点都访问太多次了，强制停止

                    # 3. 【软刹车】概率性终止 (Probabilistic Termination)
                    # 动态计算停止概率：基础概率 + (当前长度 * 0.02)
                    # 例子：长度为 5 时，停止概率 = 0.05 + 0.10 = 15%
                    # 长度为 10 时，停止概率 = 0.05 + 0.20 = 25%
                    current_stop_prob = base_stop_prob + (len(chain) * 0.01)
                    
                    # 只有当链长度至少为 5 时才允许概率停止（避免大量单步链）
                    if len(chain) >= stop_steps and random.random() < current_stop_prob:
                        break 

                    # 4. 准备权重并采样
                    weights = [self.G[curr][n].get('weight', 1.0) for n in valid_neighbors]
                    probs = self._softmax(weights, temperature)
                    
                    next_node = np.random.choice(valid_neighbors, p=probs)
                    
                    chain.append(next_node)
                    node_counts[next_node] += 1
                    curr = next_node
                
                chains.append(chain)
                
        return chains

    def save_and_report(self, chains, output_dir):
        # 去重
        unique_chains_tuple = set(tuple(c) for c in chains)
        unique_chains = [list(c) for c in unique_chains_tuple]
        
        total = len(unique_chains)
        if total == 0: return

        lengths = [len(c) for c in unique_chains]
        
        # 简单的文本直方图，帮你快速看分布
        hist_bins = [0, 5, 10, 20, 50, 100]
        hist_counts, _ = np.histogram(lengths, bins=hist_bins)
        
        print(f"📊 Report: {self.file_name}")
        print(f"  • Chains: {total} | Avg Len: {np.mean(lengths):.2f} | Max Len: {max(lengths)}")
        print(f"  • Distro: <5: {hist_counts[0]}, 5-10: {hist_counts[1]}, 10-20: {hist_counts[2]}, >20: {sum(hist_counts[3:])}")
        
        # 只有当所有链都触顶时才发出警告
        if min(lengths) == self.num_nodes:
            print(f"  ⚠️ WARNING: All chains hit max length! Increase stop_prob.")

        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"chains_{self.file_name}")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump({"meta": {"avg_len": np.mean(lengths)}, "chains": unique_chains}, f)

# --- 批量运行 ---
def process_all(input_dir, output_dir):
    files = list(Path(input_dir).glob("*.json"))
    for f in files:
        base_stop_prob = 0.1
        stop_steps = 5
        if 'math' in str(f) or 'memory' in str(f):
            base_stop_prob += 0.3
            stop_steps = 2
        print(f, stop_steps, base_stop_prob)
        gen = ToolChainGenerator(str(f))
        # 设置 max_repeats=3 防止死循环
        # 设置 base_stop_prob=0.1 (10% 基础概率停止)
        chains = gen.generate(
            max_repeats=3, 
            base_stop_prob=base_stop_prob, 
            temperature=1.2,
            stop_steps=stop_steps
        )
        gen.save_and_report(chains, output_dir)

process_all("./tool_env_graphs", "./file_output_chains")

tool_env_graphs/AutomotiveServiceRepairSystem_tool_graph.json 5 0.1
📊 Report: AutomotiveServiceRepairSystem_tool_graph.json
  • Chains: 51 | Avg Len: 8.84 | Max Len: 20
  • Distro: <5: 0, 5-10: 33, 10-20: 17, >20: 1
tool_env_graphs/CRMSystem_tool_graph.json 5 0.1
📊 Report: CRMSystem_tool_graph.json
  • Chains: 63 | Avg Len: 8.56 | Max Len: 18
  • Distro: <5: 0, 5-10: 45, 10-20: 18, >20: 0
tool_env_graphs/ChatApplicationBackend_tool_graph.json 5 0.1
📊 Report: ChatApplicationBackend_tool_graph.json
  • Chains: 60 | Avg Len: 9.58 | Max Len: 18
  • Distro: <5: 0, 5-10: 33, 10-20: 27, >20: 0
tool_env_graphs/CorporateFinancialReportingSystem_tool_graph.json 5 0.1
📊 Report: CorporateFinancialReportingSystem_tool_graph.json
  • Chains: 75 | Avg Len: 9.48 | Max Len: 22
  • Distro: <5: 0, 5-10: 40, 10-20: 34, >20: 1
tool_env_graphs/DataBackupRecoverySystem_tool_graph.json 5 0.1
📊 Report: DataBackupRecoverySystem_tool_graph.json
  • Chains: 51 | Avg Len: 8.29 | Max Len: 14
  • Distro: <5: 0, 5-10

In [ ]:
#跨环境
import networkx as nx
import numpy as np
import json
import os
import random
from collections import defaultdict
import time

# =================CONFIGURATION=================
# 每个环境作为起点生成的任务数量
NUM_SAMPLES_PER_START = 30

# 输出文件路径
OUTPUT_FILE = "file_output_chains/chains_cross_env.json"

# 数据文件夹
GRAPH_DIR = "tool_env_graphs"
# ===============================================

class ToolChainGenerator:
    def __init__(self, graph_path):
        self.graph_path = graph_path
        self.file_name = os.path.basename(graph_path)
        try:
            with open(graph_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            self.G = nx.node_link_graph(data, directed=True)
            self.nodes = list(self.G.nodes())
        except Exception as e:
            # 静默失败，避免打断批处理，但在日志中记录
            # print(f"⚠️ Error loading {self.file_name}: {e}")
            self.G = nx.DiGraph()
            self.nodes = []

    def _softmax(self, weights, temperature=1.0):
        weights = np.array(weights)
        e_x = np.exp((weights - np.max(weights)) / temperature)
        return e_x / e_x.sum()

    def generate_single_chain(self, start_node=None, max_len=5, temperature=1.0):
        if not self.nodes: return []
        
        curr = start_node
        
        # --- 需求修改点 2: 未指定起点时，优先选择入度为0的节点 ---
        if curr is None or curr not in self.G:
            # 寻找入度为0的节点 (Entry Points)
            entry_points = [n for n in self.nodes if self.G.in_degree(n) == 0]
            
            if entry_points:
                # 如果有明确入口，从中随机选一个
                curr = random.choice(entry_points)
            else:
                # 如果是强连通图(没有入度为0的点)，或者只是单纯的死循环图，则随机全图采样
                curr = random.choice(self.nodes)

        chain = [curr]
        
        # 游走生成
        # 使用动态 max_len 增加多样性，允许波动 +/- 1
        actual_limit = max(2, max_len + random.randint(-1, 2))
        
        for _ in range(actual_limit - 1):
            neighbors = list(self.G.successors(curr))
            if not neighbors: break 
            
            # 获取权重
            weights = [self.G[curr][n].get('weight', 1.0) for n in neighbors]
            probs = self._softmax(weights, temperature)
            
            next_node = np.random.choice(neighbors, p=probs)
            
            # 简单的防短环逻辑 (可选): 如果下一跳就是当前节点(自环)，降低概率重试一次
            if next_node == curr and len(neighbors) > 1:
                 next_node = np.random.choice(neighbors, p=probs)

            chain.append(next_node)
            curr = next_node
            
        return chain

class HierarchicalSampler:
    def __init__(self, base_dir):
        self.base_dir = base_dir
        
        self.env_files = {
            # ── Gen-1: 入口/能力/业务 ──
            "message": "message_tool_graph.json",
            "twitter": "twitter_tool_graph.json",
            "web": "web_search_graph.json",
            "file": "file_tool_graph.json",
            "math": "math_tool_graph.json",
            "mem_blob": "memory_blob_tool_graph.json",
            "mem_kv": "memorykv_tool_graph.json",
            "mem_vec": "vector_memory_tool_graph.json",
            "ticket": "ticketing_tool_graph.json",
            "trade": "trading_tool_graph.json",
            "travel": "travel_tool_graph.json",
            "vehicle": "vehicle_tool_graph.json",

            # ── Gen-2: OASIS 自动生成图 ──
            "travel_compliance__crud": "travel_compliance__crud_tool_graph.json",
            "travel_compliance__orchestration": "travel_compliance__orchestration_tool_graph.json",
            "travel_compliance__pipeline": "travel_compliance__pipeline_tool_graph.json",
            "travel_compliance__scheduling": "travel_compliance__scheduling_tool_graph.json",
            "travel_compliance__state_machine": "travel_compliance__state_machine_tool_graph.json",
            "travel_disruption__crud": "travel_disruption__crud_tool_graph.json",
            "travel_disruption__event_driven": "travel_disruption__event_driven_tool_graph.json",
            "travel_disruption__orchestration": "travel_disruption__orchestration_tool_graph.json",
            "travel_disruption__scheduling": "travel_disruption__scheduling_tool_graph.json",

            "investment_research__matching": "investment_research__matching_tool_graph.json",
            "investment_research__pipeline": "investment_research__pipeline_tool_graph.json",
            "investment_research__streaming": "investment_research__streaming_tool_graph.json",
            "quantitative_finance__crud": "quantitative_finance__crud_tool_graph.json",
            "quantitative_finance__pipeline": "quantitative_finance__pipeline_tool_graph.json",
            "quantitative_finance__streaming": "quantitative_finance__streaming_tool_graph.json",

            "ecommerce_logistics__crud": "ecommerce_logistics__crud_tool_graph.json",
            "ecommerce_logistics__orchestration": "ecommerce_logistics__orchestration_tool_graph.json",
            "ecommerce_logistics__pipeline": "ecommerce_logistics__pipeline_tool_graph.json",
            "ecommerce_logistics__state_machine": "ecommerce_logistics__state_machine_tool_graph.json",

            "reimbursement__crud": "reimbursement__crud_tool_graph.json",
            "reimbursement__matching": "reimbursement__matching_tool_graph.json",
            "reimbursement__orchestration": "reimbursement__orchestration_tool_graph.json",
            "reimbursement__pipeline": "reimbursement__pipeline_tool_graph.json",
            "reimbursement__state_machine": "reimbursement__state_machine_tool_graph.json",

            "security_patrol__event_driven": "security_patrol__event_driven_tool_graph.json",
            "security_patrol__orchestration": "security_patrol__orchestration_tool_graph.json",
            "security_patrol__pipeline": "security_patrol__pipeline_tool_graph.json",

            "email_automation__event_driven": "email_automation__event_driven_tool_graph.json",
            "email_automation__pipeline": "email_automation__pipeline_tool_graph.json",
            "cross_modal_transcription__conversational": "cross_modal_transcription__conversational_tool_graph.json",
            "cross_modal_transcription__pipeline": "cross_modal_transcription__pipeline_tool_graph.json",
            "industry_sentiment__orchestration": "industry_sentiment__orchestration_tool_graph.json",
            "industry_sentiment__pipeline": "industry_sentiment__pipeline_tool_graph.json",

            "personal_assistant__conversational": "personal_assistant__conversational_tool_graph.json",
            "personal_assistant__orchestration": "personal_assistant__orchestration_tool_graph.json",
            "personal_assistant__scheduling": "personal_assistant__scheduling_tool_graph.json",

            "environment_adaptive__event_driven": "environment_adaptive__event_driven_tool_graph.json",
            "environment_adaptive__streaming": "environment_adaptive__streaming_tool_graph.json",
        }
        
        self.meta_graph = self._build_meta_graph()
        # 预加载所有图数据到内存，避免每次IO，提高大规模生成速度
        self.tool_generators = {}
        self._preload_generators()

    def _preload_generators(self):
        print("⏳ Preloading graph data into memory...")
        for key, filename in self.env_files.items():
            path = os.path.join(self.base_dir, filename)
            if os.path.exists(path):
                self.tool_generators[key] = ToolChainGenerator(path)
            else:
                print(f"⚠️ Warning: {filename} missing.")
        print("✅ Preload complete.")

    def _build_meta_graph(self):
        G = nx.DiGraph()
        keys = self.env_files.keys()

        def add_flow(src, dests, weight=1.0):
            if src not in keys:
                return
            for dest in dests:
                if dest in keys:
                    G.add_edge(src, dest, weight=weight)

        # ═══════════════════════════════════════════
        # A. Gen-1 触发层 (Triggers)
        # ═══════════════════════════════════════════
        add_flow("message", [
            "web", "trade", "travel", "vehicle", "ticket", "twitter", "mem_kv",
            "email_automation__event_driven", "email_automation__pipeline",
            "cross_modal_transcription__conversational", "cross_modal_transcription__pipeline",
            "personal_assistant__conversational", "personal_assistant__orchestration",
            "personal_assistant__scheduling",
            "industry_sentiment__orchestration", "industry_sentiment__pipeline",
        ], weight=1.0)
        add_flow("twitter", ["web", "message", "industry_sentiment__orchestration"], weight=0.8)

        # ═══════════════════════════════════════════
        # B. Gen-1 能力层 (Utility)
        # ═══════════════════════════════════════════
        add_flow("web", ["file", "mem_blob", "math", "industry_sentiment__pipeline"], weight=1.5)
        add_flow("math", ["trade", "travel", "file",
                           "quantitative_finance__pipeline", "quantitative_finance__streaming"], weight=1.5)
        add_flow("mem_blob", ["file", "message"], weight=1.0)
        add_flow("mem_kv", ["trade", "travel", "ticket",
                             "personal_assistant__scheduling", "travel_compliance__scheduling"], weight=1.0)
        add_flow("mem_vec", ["message", "file", "personal_assistant__conversational"], weight=1.0)
        add_flow("file", ["message", "mem_blob",
                           "reimbursement__pipeline", "reimbursement__state_machine",
                           "ecommerce_logistics__pipeline"], weight=0.5)

        # ═══════════════════════════════════════════
        # C. Gen-1 垂直业务 → OASIS
        # ═══════════════════════════════════════════
        add_flow("vehicle", ["ticket", "message",
                              "security_patrol__event_driven", "environment_adaptive__event_driven"], weight=2.0)
        add_flow("trade", ["file", "twitter", "math",
                            "quantitative_finance__crud", "investment_research__matching"], weight=1.0)
        add_flow("travel", ["mem_kv", "twitter", "file",
                             "travel_compliance__crud", "travel_disruption__crud"], weight=1.0)
        add_flow("ticket", ["message", "file",
                             "ecommerce_logistics__crud", "reimbursement__crud",
                             "security_patrol__orchestration"], weight=1.0)

        # ═══════════════════════════════════════════
        # D. OASIS 领域内流转 (Within-Domain Flow)
        # ═══════════════════════════════════════════

        # --- travel_compliance ---
        add_flow("travel_compliance__crud", ["travel_compliance__pipeline"], 2.0)
        add_flow("travel_compliance__pipeline", ["travel_compliance__orchestration", "travel_compliance__state_machine"], 1.5)
        add_flow("travel_compliance__orchestration", ["travel_compliance__scheduling"], 1.5)
        add_flow("travel_compliance__state_machine", ["travel"], 1.0)

        # --- travel_disruption ---
        add_flow("travel_disruption__crud", ["travel_disruption__event_driven", "travel_disruption__pipeline"], 1.5)
        add_flow("travel_disruption__event_driven", ["travel_disruption__orchestration"], 2.0)
        add_flow("travel_disruption__orchestration", ["travel_disruption__scheduling", "travel"], 1.0)
        add_flow("travel_disruption__pipeline", ["travel_disruption__scheduling"], 1.0)

        # --- investment_research ---
        add_flow("investment_research__matching", ["investment_research__pipeline"], 2.0)
        add_flow("investment_research__pipeline", ["investment_research__streaming", "trade"], 1.0)

        # --- quantitative_finance ---
        add_flow("quantitative_finance__crud", ["quantitative_finance__pipeline"], 2.0)
        add_flow("quantitative_finance__pipeline", ["quantitative_finance__streaming", "math"], 1.5)
        add_flow("quantitative_finance__streaming", ["trade", "file"], 1.0)

        # --- ecommerce_logistics ---
        add_flow("ecommerce_logistics__crud", ["ecommerce_logistics__pipeline", "ecommerce_logistics__state_machine"], 2.0)
        add_flow("ecommerce_logistics__pipeline", ["ecommerce_logistics__orchestration"], 1.5)
        add_flow("ecommerce_logistics__orchestration", ["ticket"], 1.0)
        add_flow("ecommerce_logistics__state_machine", ["ticket", "message"], 1.0)

        # --- reimbursement ---
        add_flow("reimbursement__crud", ["reimbursement__pipeline", "reimbursement__state_machine"], 2.0)
        add_flow("reimbursement__pipeline", ["reimbursement__matching", "reimbursement__orchestration"], 1.5)
        add_flow("reimbursement__matching", ["reimbursement__orchestration"], 1.5)
        add_flow("reimbursement__orchestration", ["file", "message"], 1.0)

        # --- security_patrol ---
        add_flow("security_patrol__event_driven", ["security_patrol__pipeline", "security_patrol__orchestration"], 2.0)
        add_flow("security_patrol__pipeline", ["security_patrol__orchestration"], 1.5)
        add_flow("security_patrol__orchestration", ["ticket", "vehicle"], 1.0)

        # --- email_automation ---
        add_flow("email_automation__event_driven", ["email_automation__pipeline"], 2.0)
        add_flow("email_automation__pipeline", ["message", "twitter"], 1.0)

        # --- cross_modal_transcription ---
        add_flow("cross_modal_transcription__conversational", ["cross_modal_transcription__pipeline"], 2.0)
        add_flow("cross_modal_transcription__pipeline", ["file", "message"], 1.0)

        # --- industry_sentiment ---
        add_flow("industry_sentiment__orchestration", ["industry_sentiment__pipeline", "twitter"], 1.5)
        add_flow("industry_sentiment__pipeline", ["message", "web"], 1.0)

        # --- personal_assistant ---
        add_flow("personal_assistant__conversational", ["personal_assistant__orchestration"], 2.0)
        add_flow("personal_assistant__orchestration", ["personal_assistant__scheduling"], 2.0)
        add_flow("personal_assistant__scheduling", ["message", "mem_kv"], 1.0)

        # --- environment_adaptive ---
        add_flow("environment_adaptive__event_driven", ["environment_adaptive__streaming"], 2.0)
        add_flow("environment_adaptive__streaming", ["vehicle"], 1.0)

        # ═══════════════════════════════════════════
        # E. 闭环 (Closing)
        # ═══════════════════════════════════════════
        add_flow("ticket", ["message"], 1.0)
        add_flow("travel_compliance__scheduling", ["message"], 1.0)
        add_flow("travel_disruption__scheduling", ["message"], 0.8)
        add_flow("reimbursement__state_machine", ["message"], 1.0)
        add_flow("ecommerce_logistics__state_machine", ["message"], 1.0)
        add_flow("security_patrol__pipeline", ["message"], 1.0)
        add_flow("investment_research__streaming", ["twitter"], 0.8)

        return G

    def sample_task(self, start_env, max_envs, temperature_noise=0.0):
        """
        todo docs
        """
        # 1. Meta-Path Sampling
        env_path = [start_env]
        curr = start_env
        
        # 随机游走长度
        target_len = random.randint(2, max_envs)
        
        for _ in range(target_len - 1):
            if curr not in self.meta_graph: break
            neighbors = list(self.meta_graph.successors(curr))
            if not neighbors: break
            
            # 基础权重
            weights = [self.meta_graph[curr][n]['weight'] for n in neighbors]
            total = sum(weights)
            probs = [w/total for w in weights]
            
            next_env = random.choices(neighbors, weights=probs, k=1)[0]
            
            # 去重：尽量不连续
            if next_env == curr: continue
            env_path.append(next_env)
            curr = next_env

        # 2. Tool-Chain Generation
        full_chain_data = []
        
        for env_key in env_path:
            gen = self.tool_generators.get(env_key)
            if not gen: continue
            
            # --- 需求修改点 3: 增加多样性 (参数随机化) ---
            # 基础步数根据环境类型不同
            if env_key in ["math", "mem_kv", "mem_vec"]:
                base_steps = 3
            elif env_key in ["web", "file"]:
                base_steps = 4
            else:
                base_steps = 5
            
            # 引入随机 Temperature (0.8 ~ 1.4)
            step_temp = temperature_noise
            
            # 生成子链
            sub_chain = gen.generate_single_chain(max_len=base_steps, temperature=step_temp)
            
            # 添加
            labeled_chain = [f"[{env_key}] {tool}" for tool in sub_chain]
            full_chain_data.extend(labeled_chain)
            
        return env_path, full_chain_data

# =================MAIN EXECUTION=================

def main():
    if not os.path.exists(GRAPH_DIR):
        print(f"❌ Error: Directory '{GRAPH_DIR}' not found.")
        return

    sampler = HierarchicalSampler(base_dir=GRAPH_DIR)
    
    all_tasks = []
    task_counter = 0
    start_time = time.time()

    print(f"🚀 Starting generation. Target: {len(sampler.env_files)} envs * {NUM_SAMPLES_PER_START} samples = {len(sampler.env_files)*NUM_SAMPLES_PER_START} tasks")
    print("-" * 50)

    # --- 需求修改点 1: 遍历每个环境作为起点 ---
    for start_env in sampler.env_files.keys():
        print(f"📍 Processing Start Node: {start_env} ...")
        
        for i in range(NUM_SAMPLES_PER_START):
            # 随机化最大环境跳数 (2-5之间)
            max_hops = random.randint(2, 6) 
            
            # 采样
            path, tools = sampler.sample_task(
                start_env=start_env, 
                max_envs=max_hops,
                temperature_noise=1.0
            )
            
            if len(tools) > 0:
                all_tasks.append({
                    "id": task_counter,
                    "start_env": start_env,
                    "meta_path_len": len(path),
                    "meta_path": path,
                    "tool_chain_len": len(tools),
                    "tool_chain": tools
                })
                task_counter += 1

    duration = time.time() - start_time
    print("-" * 50)
    print(f"✅ Generation Complete in {duration:.2f}s")
    print(f"📊 Total Unique Chains: {len(all_tasks)}")
    
    # 简单的分布统计
    avg_len = sum(t['tool_chain_len'] for t in all_tasks) / len(all_tasks)
    print(f"📊 Avg Chain Length: {avg_len:.2f}")

    with open(OUTPUT_FILE, "w", encoding='utf-8') as f:
        json.dump(all_tasks, f, indent=2)
    print(f"💾 Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()